#Cuaderno de Esteban Boroni para el Challenge de la empresa Drone AI Services.

##Instalación de dependencias

In [16]:
!pip install ultralytics pycocotools -q

##Importación de librerías

In [17]:
import os
import json
import shutil
import yaml
from pathlib import Path
from ultralytics import YOLO
from ultralytics.data.converter import convert_coco

#Borrar las celdas de debajo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Descomprimimos el ZIP desde Drive hacia la carpeta temporal /content/dataset_dais
!unzip "/content/drive/MyDrive/skids_detection_challenge" -d "/content/dataset_dais"

Archive:  /content/drive/MyDrive/skids_detection_challenge.zip
replace /content/dataset_dais/skids_detection/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset_dais/skids_detection/test/img_00001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset_dais/skids_detection/test/img_00002.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset_dais/skids_detection/test/img_00009.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset_dais/skids_detection/test/img_00013.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset_dais/skids_detection/test/img_00026.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

##Preprocesamiento de los datos

##Cambiar estructura del dataset para convertir a YOLO

In [19]:
source_dir = "/content/dataset_dais/skids_detection"
target_dir = "/content/my_dataset"

# 2. Mapeo de carpetas origen -> destino
# (Transforma 'valid' en 'val' como pediste, y mantiene 'test')
splits = {
    "train": "train",
    "valid": "val",
    "test": "test"
}

# Crear carpetas principales de destino
os.makedirs(os.path.join(target_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(target_dir, "annotations"), exist_ok=True)

for src_split, dst_split in splits.items():
    src_split_path = os.path.join(source_dir, src_split)

    if not os.path.exists(src_split_path):
        print(f"Advertencia: No se encontró la carpeta {src_split_path}")
        continue

    # Crear carpeta de imágenes para el split (ej: my_dataset/images/train)
    dst_images_path = os.path.join(target_dir, "images", dst_split)
    os.makedirs(dst_images_path, exist_ok=True)

    print(f"Copiando conjunto '{src_split}' -> '{dst_split}'...")

    for filename in os.listdir(src_split_path):
        src_file = os.path.join(src_split_path, filename)

        if not os.path.isfile(src_file):
            continue

        if filename == "_annotations.coco.json":
            # Si es el JSON, va a la carpeta de anotaciones con el nuevo nombre
            dst_ann_path = os.path.join(target_dir, "annotations", f"instances_{dst_split}.json")
            shutil.copy2(src_file, dst_ann_path)
        else:
            # Si es una imagen, va a la carpeta images/split/
            dst_img_path = os.path.join(dst_images_path, filename)
            shutil.copy2(src_file, dst_img_path)

Copiando conjunto 'train' -> 'train'...
Copiando conjunto 'valid' -> 'val'...
Copiando conjunto 'test' -> 'test'...


In [20]:
convert_coco(
    labels_dir="/content/my_dataset/annotations/",
    save_dir="/content/my_dataset/converted/",
    cls91to80=False,
)

Annotations /content/my_dataset/annotations/instances_test.json: 100% ━━━━━━━━━━━━ 269/269 3.5Kit/s 0.1s
Annotations /content/my_dataset/annotations/instances_train.json: 100% ━━━━━━━━━━━━ 1164/1164 4.2Kit/s 0.3s
Annotations /content/my_dataset/annotations/instances_val.json: 100% ━━━━━━━━━━━━ 249/249 4.8Kit/s 0.1s
COCO data converted successfully.
Results saved to /content/my_dataset/converted-3


In [21]:
converted_dir = Path("/content/my_dataset/converted/labels")
dataset_dir = Path("/content/my_dataset")

# convert_coco names each subdirectory after its JSON file (minus the "instances_" prefix),
# so iterate the actual subdirectories instead of assuming "train"/"val".
for src in converted_dir.iterdir():
    if not src.is_dir():
        continue
    dst = dataset_dir / "labels" / src.name
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.glob("*.txt"):
        shutil.move(str(f), str(dst / f.name))

In [22]:
# Read categories from your COCO JSON
with open("/content/my_dataset/annotations/instances_train.json") as f:
    coco = json.load(f)

# Generar los nombres ignorando la clase dummy (id 0 o nombre 'objects')
names = {}
for cat in coco["categories"]:
    if cat["id"] == 0 or cat["name"] == "objects":
        continue # Saltamos la categoría sin instancias

    # convert_coco mapea los IDs restando 1, así que hacemos lo mismo en el YAML
    names[cat["id"] - 1] = cat["name"]

# Create dataset.yaml
dataset = {
    "path": str(Path("/content/my_dataset").resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test", # Agregado por si querés correr validación en el test set
    "names": names,
}

with open("/content/my_dataset/dataset.yaml", "w") as f:
    # sort_keys=False asegura que los IDs queden en orden (0, 1, 2) en el archivo
    yaml.dump(dataset, f, default_flow_style=False, sort_keys=False)

##Entrenamiento del modelo

In [ ]:
model = YOLO('yolov8s.pt') # 's' es de Small. Podrías usar 'n' (Nano) o 'm' (Medium)

# 2. Iniciar el entrenamiento
results = model.train(
    data='/content/my_dataset/dataset.yaml',
    project='Drone_AI_Challenge',
    name='yolov8s_skids_run1',

    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    device=0,

    # --- Hiperparámetros base ---
    optimizer='auto',
    lr0=0.01,
    cos_lr=True,

    save=True,
    plots=True
)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/my_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_skids_run1-2, nbs=64, nms=None, ops

In [ ]:
origen = "/content/runs/detect/Drone_AI_Challenge/yolov8s_skids_run1-2/weights/best.pt"
destino = "/content/drive/MyDrive/yolov8s_best_skids.pt"

shutil.copy(origen, destino)

'/content/drive/MyDrive/yolov8s_best_skids.pt'

##Prueba sobre el conjunto Test

In [23]:
# Cargar tu modelo recién guardado
best_model = YOLO('/content/drive/MyDrive/yolov8s_best_skids.pt')

# Predecir sobre la carpeta de test
results = best_model.predict(
    source='/content/my_dataset/images/test',
    conf=0.35, # Umbral de confianza del 35%
    save=True, # Dibuja las cajas y guarda las fotos
    project='Drone_AI_Challenge',
    name='predicciones_test'
)


image 1/534 /content/my_dataset/images/test/img_00001.jpg: 640x640 1 Skid, 958.2ms
image 2/534 /content/my_dataset/images/test/img_00002.jpg: 640x640 1 Skid, 760.7ms
image 3/534 /content/my_dataset/images/test/img_00009.jpg: 640x640 1 Skid, 474.5ms
image 4/534 /content/my_dataset/images/test/img_00013.jpg: 640x640 1 Skid, 509.6ms
image 5/534 /content/my_dataset/images/test/img_00026.jpg: 640x640 1 Skid, 480.7ms
image 6/534 /content/my_dataset/images/test/img_00045.jpg: 640x640 1 Skid, 497.0ms


KeyboardInterrupt: 

In [12]:
print("Evaluando sobre el conjunto de Test...")
metrics = best_model.val(
    data='/content/my_dataset/dataset.yaml',
    split='test',
    project='Drone_AI_Challenge',
    name='metricas_test'
)

# 3. Extraer y mostrar las métricas por clase para analizar el desbalance
print("\n" + "="*60)
print(f"{'Clase':<12} | {'F1-Score':<9} | {'Precision':<9} | {'Recall':<9} | {'mAP50':<5}")
print("="*60)

nombres_clases = metrics.names

for i, clase_id in enumerate(metrics.box.ap_class_index):
    nombre = nombres_clases[clase_id]

    # Extraemos Precisión y Recall para la clase actual
    p = metrics.box.p[i]
    r = metrics.box.r[i]

    # Calculamos el F1-Score
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

    # CORRECCIÓN: Usamos ap50[i] para obtener el valor específico de esta clase
    ap50 = metrics.box.ap50[i]

    print(f"{nombre:<12} | {f1:<9.3f} | {p:<9.3f} | {r:<9.3f} | {ap50:<5.3f}")

print("="*60)
print(f"Las curvas gráficas se guardaron en: {metrics.save_dir}")

Evaluando sobre el conjunto de Test...
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1395.5±776.6 MB/s, size: 135.2 KB)
val: Scanning /content/my_dataset/labels/test... 269 images, 265 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 534/534 2.2Kit/s 0.2s
val: New cache created: /content/my_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 10.4s/it 5:54
                   all        534        367      0.829      0.729        0.8      0.624
                  Skid        189        226      0.899      0.907       0.94      0.759
              Volquete         71        128      0.823      0.742       0.77      0.583
                  Bano          9         13      0.766      0.538       0.69      0.531
Speed: 6.9ms preprocess, 646.4ms i

In [15]:
origen = '/content/runs/detect/Drone_AI_Challenge/metricas_test'
destino = '/content/drive/MyDrive/Drone_AI_Metricas_Test'

shutil.copytree(origen, destino, dirs_exist_ok=True)

'/content/drive/MyDrive/Drone_AI_Metricas_Test'